In [ ]:
# ============================================================
# CELL 0: CHỌN SEED  (CHỈ SỬA DÒNG NÀY)
# ============================================================
SEED = 0   # chạy seed 0..19, trùng seed của mô hình gốc và TSVM
print(f"C2PSA rộng: SEED={SEED}")


In [ ]:
# ============================================================
# CELL 1: NẠP ULTRALYTICS C2PSA RỘNG + DATA_BG20.YAML
# ============================================================
import os, sys

# Tên tài khoản Kaggle đang giữ dataset; notebook tự tìm theo thứ tự trong danh sách.
KAGGLE_USERS = ["okeydokey0805", "huyphngtun"]

def _input_path(slug, sub, marker):
    # Kaggle gắn dataset ở /kaggle/input/datasets/<user>/<slug> hoặc /kaggle/input/<slug>,
    # có thể giải nén thẳng vào gốc dataset (không còn thư mục con <sub>)
    bases = [f"/kaggle/input/datasets/{u}/{slug}" for u in KAGGLE_USERS] + [f"/kaggle/input/{slug}"]
    cands = [f"{b}{t}" for b in bases for t in (f"/{sub}", "")]
    return next((p for p in cands if os.path.exists(f"{p}/{marker}")), cands[0])

# Dataset tải lên từ 02_ma_nguon/ultralytics_C2PSA_rong (đặt slug là ultralytics-c2psa-rong)
CUSTOM_REPO_PATH = _input_path("ultralytics-c2psa-rong", "ultralytics_C2PSA_rong", "C2PSA_RONG.md")
DATASET_ROOT = _input_path("kvasir-yolo-seg-bg20", "Kvasir_YOLO_SEG_BG20", "images/train")
assert os.path.exists(f"{CUSTOM_REPO_PATH}/C2PSA_RONG.md"), f"❌ Không tìm thấy repo C2PSA rộng: {CUSTOM_REPO_PATH}"
assert os.path.exists(f"{DATASET_ROOT}/images/train"), f"❌ Không tìm thấy dataset: {DATASET_ROOT}"
print("📁 Repo:", CUSTOM_REPO_PATH)
print("📁 Dữ liệu:", DATASET_ROOT)

if not os.path.exists("/kaggle/working/ultralytics"):
    os.symlink(CUSTOM_REPO_PATH, "/kaggle/working/ultralytics")
    print("🔗 Đã tạo symlink 'ultralytics'")
for p in ["/kaggle/working", CUSTOM_REPO_PATH]:
    if p not in sys.path:
        sys.path.insert(0, p)

import ultralytics, yaml
print(f"✅ Nạp Ultralytics ({ultralytics.__version__}) từ: {ultralytics.__file__}")

# Tên tệp bắt đầu bằng "yolo26s" để Ultralytics chọn scale s
YAML_C2PSA = f"{CUSTOM_REPO_PATH}/cfg/models/26/yolo26s-seg-C2PSA-rong.yaml"
with open(YAML_C2PSA, encoding="utf-8") as f:
    print("✅ Tầng 10:", yaml.safe_load(f)["backbone"][10])

DATA_YAML = "/kaggle/working/data_bg20.yaml"
with open(DATA_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump({"path": DATASET_ROOT, "train": "images/train", "val": "images/val",
                    "nc": 1, "names": {0: "polyp"}}, f, sort_keys=False, allow_unicode=True)
print("✅ Đã tạo data_bg20.yaml")


In [ ]:
# ============================================================
# CELL 2: TẢI SẴN FILE TRỌNG SỐ (CÓ TỰ ĐỘNG RETRY & KIỂM TRA DUNG LƯỢNG)
# ============================================================
import os

WEIGHT_PATH = "/kaggle/working/yolo26s-seg.pt"
URL = "https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo26s-seg.pt"

if os.path.exists(WEIGHT_PATH) and os.path.getsize(WEIGHT_PATH) < 20 * 1024 * 1024:
    os.remove(WEIGHT_PATH)

if not os.path.exists(WEIGHT_PATH):
    print("⏳ Đang tải trọng số yolo26s-seg.pt...")
    !curl -L --retry 5 --retry-delay 2 -C - "{URL}" -o "{WEIGHT_PATH}"

size_mb = os.path.getsize(WEIGHT_PATH) / (1024 * 1024)
print(f" Dung lượng file: {size_mb:.2f} MB")
assert size_mb > 20.0, "❌ File tải về chưa đủ dung lượng, hãy chạy lại cell này! (Settings > Internet: ON)"

In [ ]:
# ============================================================
# CELL 3: KIỂM TRA MÔ HÌNH TRƯỚC KHI TRAIN (~10 GIÂY)
# ============================================================
from ultralytics import YOLO

EXPECTED_L10 = 1_830_016   # C2PSA rộng (e = 0,75); TSVM là 1.811.201, C2PSA gốc là 990.976
_m = YOLO(YAML_C2PSA)
l10 = _m.model.model[10]
n_l10 = sum(p.numel() for p in l10.parameters())
print(f"Tầng 10: {type(l10).__name__}, số khối = {len(l10.m)}, tham số = {n_l10:,} (mong đợi {EXPECTED_L10:,})")
assert type(l10).__name__ == "C2PSA", "❌ Tầng 10 không phải C2PSA"
assert n_l10 == EXPECTED_L10, "❌ Số tham số tầng 10 khác mong đợi, kiểm tra lại YAML"
print("✅ Mô hình C2PSA rộng đúng cấu trúc")
del _m


In [ ]:
# ============================================================
# CELL 4: KHÓA TẤT ĐỊNH & TRAIN C2PSA RỘNG (BG20 - 100 EPOCHS) - GIỐNG HỆT 20 SEED TSVM FP32
# ============================================================
import os, random, numpy as np, torch
from ultralytics import YOLO

WORKERS = 2
BATCH = 8
EPOCHS = 100

os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True, warn_only=False)

# Tầng 10 khác kích thước với yolo26s-seg.pt nên học từ đầu (giống TSVM); các tầng khác nạp trọng số có sẵn
model = YOLO(YAML_C2PSA).load(WEIGHT_PATH)

RUN_NAME = f"Kvasir_BG20_YOLO26s_seg_C2PSA_rong_s{SEED}_w{WORKERS}"
results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=640,
    batch=BATCH,
    device=0,
    workers=WORKERS,
    cache=False,
    amp=False,
    optimizer="AdamW",
    seed=SEED,
    lr0=0.001,
    warmup_epochs=5.0,
    deterministic=True,
    patience=100,
    close_mosaic=10,
    project="/kaggle/working/runs",
    name=RUN_NAME,
    exist_ok=True,
    plots=True
)

In [ ]:
# ============================================================
# CELL 5: NÉN KẾT QUẢ ĐỂ TẢI VỀ
# ============================================================
!cd /kaggle/working && zip -r "runs_{RUN_NAME}.zip" runs